In [1]:
import quarters_clean

In [2]:
gov_expenditure_df = quarters_clean.pd.read_parquet("../parquet_exports/government_expenditure_values.parquet")

In [3]:
gov_expenditure_df.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,1,1960,0.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9.43,NaN,NaN,NaN,NaN,NaN,29.35,1.02,NaN
1,2,1960,0.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,8.79,NaN,NaN,NaN,NaN,NaN,29.73,1.05,NaN
2,3,1960,0.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,8.35,NaN,NaN,NaN,NaN,NaN,30.73,1.05,NaN
3,4,1960,0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9.00,NaN,NaN,NaN,NaN,NaN,31.23,1.08,NaN
4,1,1961,0.48,NaN,NaN,NaN,NaN,1.75,NaN,NaN,...,NaN,9.81,NaN,NaN,NaN,NaN,NaN,31.70,1.12,NaN


In [4]:
gov_expenditure_df.tail()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
257,2,2024,156.31,27.03,37.45,545.21,9.65,161.12,11818.46,62335.87,...,345.79,114008.8,79.09,421.86,23.78,744.67,1465.73,1235.75,147.51,9957.51
258,3,2024,155.21,26.19,34.95,535.10,9.65,165.91,11518.06,58028.78,...,354.34,108693.2,72.50,414.51,23.95,856.67,1618.97,1258.75,149.82,10463.28
259,4,2024,161.47,28.77,39.64,660.17,11.59,173.31,12467.10,86802.11,...,362.05,111135.2,85.91,453.26,24.27,790.15,1947.17,1274.93,156.07,12498.37
260,1,2025,157.62,26.20,37.32,517.27,9.55,179.13,12137.12,51936.80,...,351.54,118271.4,73.77,412.74,24.36,747.59,1884.21,1287.48,157.65,9983.01
261,2,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,81.73,NaN,NaN,NaN,NaN,1297.43,NaN,NaN


In [5]:
quarters_clean.summarize_missing_by_country(gov_expenditure_df)

,Missing Count,Missing %
India,206,78.63
Malaysia,189,72.14
Macedonia,161,61.45
Colombia,161,61.45
Turkey,153,58.40
Chile,145,55.34
Brazil,145,55.34
Russia,143,54.58
Israel,141,53.82
Ireland,141,53.82


**H1)** The previous output indicates that India will be the country that will most restrict us in the time range available to perform the correlation analysis. This statement is only correct due to the process of extracting and joining the values of this indicator. If indicators were downloaded for a country and those values were not available for a time range already existing in the master data frame, NaN values were assigned.

**H2)** The important thing is to first cross-reference the countries available in the house price index with the countries available in this data frame in such a way as to concentrate the work on the countries that will be available and then find a similarity with Paraguay.

**O1)** The algorithm for crawling pages and obtaining data already took into account only downloading countries with available house price index values. Paraguay was also added.

In [6]:
missing_entries = quarters_clean.get_missing_entries(gov_expenditure_df)
missing_entries.head()

,Quarter,Year,Country,Value
0,1,1960,Austria,NaN
1,1,1960,Belgium,NaN
2,1,1960,Brazil,NaN
3,1,1960,Bulgaria,NaN
4,1,1960,Canada,NaN


In [8]:
india_missing = quarters_clean.clean_country_missing_entries(missing_entries, "India")
india_missing

,Quarter,Year,Country,Value
0,1,1960,India,NaN
1,2,1960,India,NaN
2,3,1960,India,NaN
3,4,1960,India,NaN
4,1,1961,India,NaN
...,...,...,...,...
201,2,2010,India,NaN
202,3,2010,India,NaN
203,4,2010,India,NaN
204,1,2011,India,NaN


The previous outputs show us that India has no values for the house price index from 19960 to 2011 nor for 2025. Consequently, it has values in the range from 2012 to 2024, a desirable range since other values which we extracted with the crawlers are defined between 2014 and 2023.

### Let's delete all the data from the beginning until 2010

In [9]:
clean_df1 = gov_expenditure_df.copy()

In [13]:
clean_df1 = quarters_clean.filter_by_year(clean_df1, min_year=2011, max_year=2024, exclude_years=[2025])
clean_df1.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,1,2012,66.77,15.17,23.24,194.54,2.91,97.81,3577.14,18888.24,...,162.67,53610.5,49.27,226.18,18.08,466.12,47.28,792.90,90.60,3689.91
1,2,2012,76.87,15.96,23.48,215.51,3.17,94.39,3864.97,21502.26,...,160.25,51743.6,54.92,239.91,18.14,503.10,55.69,789.90,86.94,4044.95
2,3,2012,70.11,15.58,22.25,215.55,3.08,94.20,3801.22,20613.12,...,169.76,52381.0,47.47,233.30,18.25,559.63,55.96,789.90,87.93,4071.74
3,4,2012,72.83,16.97,24.84,266.58,3.88,98.37,4777.10,28811.38,...,178.53,53757.7,53.32,262.56,18.36,491.94,64.47,785.88,89.90,5456.24
4,1,2013,68.90,15.51,23.84,214.49,3.19,100.59,3779.75,20802.88,...,177.69,56434.9,47.10,235.86,18.44,482.39,57.30,780.18,89.31,4184.21


In [14]:
clean_df1.tail()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
47,4,2023,148.37,26.45,37.94,630.48,11.18,160.13,11974.47,77136.89,...,356.08,105614.8,82.40,441.49,23.90,737.29,1141.85,1203.80,144.59,11413.76
48,1,2024,146.68,24.78,35.80,469.85,9.04,165.63,11150.89,46911.24,...,348.64,113219.3,71.05,404.94,23.81,709.51,1368.60,1220.25,145.11,9486.11
49,2,2024,156.31,27.03,37.45,545.21,9.65,161.12,11818.46,62335.87,...,345.79,114008.8,79.09,421.86,23.78,744.67,1465.73,1235.75,147.51,9957.51
50,3,2024,155.21,26.19,34.95,535.10,9.65,165.91,11518.06,58028.78,...,354.34,108693.2,72.50,414.51,23.95,856.67,1618.97,1258.75,149.82,10463.28
51,4,2024,161.47,28.77,39.64,660.17,11.59,173.31,12467.10,86802.11,...,362.05,111135.2,85.91,453.26,24.27,790.15,1947.17,1274.93,156.07,12498.37


In [15]:
quarters_clean.summarize_missing_by_country(clean_df1)

,Missing Count,Missing %
New-Zealand,7,13.46
Russia,1,1.92
Australia,0,0.00
Austria,0,0.00
Bulgaria,0,0.00
Canada,0,0.00
Belgium,0,0.00
Brazil,0,0.00
Croatia,0,0.00
Cyprus,0,0.00


In [20]:
missing_entries_clean_df1 = quarters_clean.get_missing_entries(clean_df1)
missing_entries_clean_df1.sort_values(by=["Year", "Quarter"], ascending=True)

,Quarter,Year,Country,Value
0,1,2012,New-Zealand,NaN
2,2,2012,New-Zealand,NaN
4,3,2012,New-Zealand,NaN
6,4,2012,New-Zealand,NaN
1,1,2013,New-Zealand,NaN
3,2,2013,New-Zealand,NaN
5,3,2013,New-Zealand,NaN
7,4,2024,Russia,NaN


In [21]:
clean_df2 = quarters_clean.filter_by_year(clean_df1, min_year=2014, max_year=2024, exclude_years=[2025])
clean_df2.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Colombia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
0,1,2015,74.24,16.46,24.34,266.32,3.26,105.98,4830.36,24532.05,...,202.50,63297.8,48.88,256.04,19.21,534.91,66.87,794.05,93.60,4972.07
1,2,2015,84.34,17.16,24.82,290.18,3.75,101.76,5163.56,27793.57,...,203.30,63427.1,54.74,271.90,19.22,573.00,77.90,804.95,92.66,5183.74
2,3,2015,79.27,16.68,23.23,287.92,3.28,102.66,5201.14,28848.05,...,213.06,60449.3,47.87,266.58,19.25,642.49,79.19,808.75,94.47,5267.48
3,4,2015,82.94,18.55,25.98,341.36,4.12,105.17,6365.15,38014.33,...,220.43,62913.8,57.83,301.07,19.39,602.65,100.59,811.18,93.91,6823.02
4,1,2016,78.65,16.96,24.73,284.75,3.22,108.39,5177.34,25092.87,...,222.17,67125.0,49.43,269.49,19.42,592.27,86.45,815.68,95.38,5299.22


In [23]:
quarters_clean.summarize_missing_by_country(clean_df2)

,Missing Count,Missing %
Russia,1,2.5
Australia,0,0.0
Belgium,0,0.0
Austria,0,0.0
Bulgaria,0,0.0
Canada,0,0.0
Chile,0,0.0
Brazil,0,0.0
Croatia,0,0.0
Cyprus,0,0.0


In [33]:
clean_df2["Russia"] = clean_df2["Russia"].fillna(method="ffill")

C:\Users\lucas\AppData\Local\Temp\ipykernel_15508\3524377934.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  clean_df2["Russia"] = clean_df2["Russia"].fillna(method="ffill")


In [34]:
quarters_clean.summarize_missing_by_country(clean_df2)

,Missing Count,Missing %
Australia,0,0.0
Austria,0,0.0
Belgium,0,0.0
Brazil,0,0.0
Bulgaria,0,0.0
Canada,0,0.0
Chile,0,0.0
Colombia,0,0.0
Croatia,0,0.0
Cyprus,0,0.0


In [35]:
clean_df2[["Quarter", "Year", "Russia"]].tail()

,Quarter,Year,Russia
35,4,2023,8325.1
36,1,2024,8664.8
37,2,2024,8783.7
38,3,2024,9321.5
39,4,2024,9321.5


In [36]:
gov_expenditure_df[["Quarter", "Year", "Russia"]].tail()

,Quarter,Year,Russia
257,2,2024,8783.7
258,3,2024,9321.5
259,4,2024,NaN
260,1,2025,NaN
261,2,2025,NaN


In [39]:
clean_df2.to_parquet("../data/government_expenditure_values.parquet", index=False)